In [1]:
import pandas as pd
df = pd.read_csv("h2.csv")

In [2]:
df.shape

(79330, 31)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79330 entries, 0 to 79329
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   IsCanceled                   79330 non-null  int64  
 1   LeadTime                     79330 non-null  int64  
 2   ArrivalDateYear              79330 non-null  int64  
 3   ArrivalDateMonth             79330 non-null  object 
 4   ArrivalDateWeekNumber        79330 non-null  int64  
 5   ArrivalDateDayOfMonth        79330 non-null  int64  
 6   StaysInWeekendNights         79330 non-null  int64  
 7   StaysInWeekNights            79330 non-null  int64  
 8   Adults                       79330 non-null  int64  
 9   Children                     79326 non-null  float64
 10  Babies                       79330 non-null  int64  
 11  Meal                         79330 non-null  object 
 12  Country                      79306 non-null  object 
 13  MarketSegment   

In [4]:
df.isnull().sum()[df.isnull().sum() > 0]

Children     4
Country     24
dtype: int64

In [5]:
df["Country"] = df["Country"].fillna(df["Country"].mode()[0])
df["Children"] = df["Children"].fillna(df["Children"].mean())

In [6]:
threshold = 0.0001 * len(df)  

agent_counts = df["Agent"].value_counts()
rare_agents = agent_counts[agent_counts < threshold].index
df["Agent"] = df["Agent"].replace(rare_agents, "Other")
print(rare_agents.shape)
 
country_counts = df["Country"].value_counts()
rare_agents = country_counts[country_counts < threshold].index
df["Country"] = df["Country"].replace(rare_agents, "Other")
print(rare_agents.shape)

Company_counts = df["Company"].value_counts()
rare_agents = Company_counts[Company_counts < threshold].index
df["Company"] = df["Company"].replace(rare_agents, "Other")
print(rare_agents.shape)

(83,)
(70,)
(143,)


In [7]:
df["DepositType"] = df["DepositType"].str.strip()

In [8]:
print(df["DepositType"].unique())
print(df["CustomerType"].unique())

['No Deposit' 'Non Refund' 'Refundable']
['Transient' 'Transient-Party' 'Contract' 'Group']


In [9]:
df.drop(["ReservationStatus", "ReservationStatusDate"], axis=1, inplace=True)

In [10]:
subset = df[
    (df["Adults"] == 0) &
    ((df["Children"] > 0) | (df["Babies"] > 0))
]

cancelled = (subset["IsCanceled"] == 1).sum()
total = len(subset)

print("Cancelled:", cancelled)
print("Total:", total)
print("Cancellation Percentage:", round(cancelled / total * 100, 2), "%")

Cancelled: 84
Total: 223
Cancellation Percentage: 37.67 %


In [11]:
df = df[~(
    (df["Adults"] == 0) &
    (df["Children"] == 0) &
    (df["Babies"] == 0)
)]

df = df[df["Adults"] > 0]

In [12]:
df.shape

(78940, 29)

In [13]:
df.shape

(78940, 29)

In [15]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(num_cols)
print(cat_cols)

['IsCanceled', 'LeadTime', 'ArrivalDateYear', 'ArrivalDateWeekNumber', 'ArrivalDateDayOfMonth', 'StaysInWeekendNights', 'StaysInWeekNights', 'Adults', 'Children', 'Babies', 'IsRepeatedGuest', 'PreviousCancellations', 'PreviousBookingsNotCanceled', 'BookingChanges', 'DaysInWaitingList', 'ADR', 'RequiredCarParkingSpaces', 'TotalOfSpecialRequests']
['ArrivalDateMonth', 'Meal', 'Country', 'MarketSegment', 'DistributionChannel', 'ReservedRoomType', 'AssignedRoomType', 'DepositType', 'Agent', 'Company', 'CustomerType']


In [16]:
ohe_cols = [
    "ArrivalDateMonth",
    "Meal",
    "Country",
    "MarketSegment",
    "DistributionChannel",
    "ReservedRoomType",
    "AssignedRoomType",
    "Agent",
    "Company"
]
ordinal_cols = [
    "DepositType",
    "CustomerType"
]
deposit_order = ["No Deposit", "Refundable", "Non Refund"]
customer_order = ["Transient", "Contract", "Transient-Party", "Group"]

In [17]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

ohe_df = pd.DataFrame(
    ohe.fit_transform(df[ohe_cols]),
    columns=ohe.get_feature_names_out(ohe_cols),
    index=df.index
)
df = pd.concat([df.drop(columns=ohe_cols), ohe_df], axis=1)

In [18]:
deposit_order = ["No Deposit", "Refundable", "Non Refund"]
customer_order = ["Transient", "Contract", "Transient-Party", "Group"]

ord_enc = OrdinalEncoder(categories=[deposit_order, customer_order])

df[["DepositType", "CustomerType"]] = ord_enc.fit_transform(
    df[["DepositType", "CustomerType"]]
)

In [19]:
df.shape

(78940, 368)

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

cv_scores = cross_val_score(
    lr,
    X_train_scaled,
    y_train,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

print("\nCV Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

Accuracy : 0.8301241449201926
Precision: 0.8313314160040602
Recall   : 0.7446582815578118
F1 Score : 0.7856115107913669
ROC-AUC  : 0.912821702474039

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      9189
           1       0.83      0.74      0.79      6599

    accuracy                           0.83     15788
   macro avg       0.83      0.82      0.82     15788
weighted avg       0.83      0.83      0.83     15788


Confusion Matrix:

[[8192  997]
 [1685 4914]]

CV Scores: [0.82685456 0.82606286 0.81631037 0.82153603 0.82367379]
Mean CV Accuracy: 0.82288752308115


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

cv_scores = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

print("\nCV Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

Accuracy : 0.8934633899163922
Precision: 0.9020441537203597
Recall   : 0.835884224882558
F1 Score : 0.8677048922447695
ROC-AUC  : 0.9596377851582727

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.93      0.91      9189
           1       0.90      0.84      0.87      6599

    accuracy                           0.89     15788
   macro avg       0.90      0.89      0.89     15788
weighted avg       0.89      0.89      0.89     15788


Confusion Matrix:

[[8590  599]
 [1083 5516]]

CV Scores: [0.88829071 0.88607395 0.88558987 0.8861441  0.88836105]
Mean CV Accuracy: 0.886891934051332


In [29]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = df.drop("IsCanceled", axis=1)
y = df["IsCanceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

Accuracy: 0.8710412971877375

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.92      0.89      9189
           1       0.88      0.80      0.84      6599

    accuracy                           0.87     15788
   macro avg       0.87      0.86      0.87     15788
weighted avg       0.87      0.87      0.87     15788


Confusion Matrix:
 [[8443  746]
 [1290 5309]]
ROC-AUC  : 0.912821702474039


In [21]:
from xgboost import XGBClassifier

xgb_cv = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    xgb_cv,
    X,
    y,
    cv=skf,
    scoring="accuracy"
)

print(scores)
print(scores.mean())

[0.87053458 0.86901444 0.86819103 0.86819103 0.87091462]
0.869369141119838
